# Phase 7 & 8: Downstream Evaluation & Reporting on Google Colab

This notebook evaluates the pretrained MoCo-v2 checkpoints across all mandatory protocols:
1. **Linear Probe**: Evaluating representation quality on frozen backbones.
2. **Label Efficiency Curves**: 1%, 5%, 10%, 25%, 50%, 100% of labels across 3 seeds.
3. **Recording-ID Leakage Control**: Verifying the representation does not cheat on recording identity.
4. **Visualizations**: Dual t-SNE (class vs recording), Grad-CAM on spectrograms, and label-efficiency curves.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 2. Clone repository & install dependencies
%cd /content
!rm -rf deepsonar_v2
!git clone https://github.com/Adhi-1004/deepsonar_v2.git
%cd /content/deepsonar_v2
!bash scripts/00_setup_env.sh

# Link checkpoints from Google Drive
!mkdir -p checkpoints
!rm -rf checkpoints
!ln -s "/content/drive/MyDrive/deepsonar_v2_checkpoints" checkpoints
!ls -lh checkpoints/*

In [ ]:
# 3. Ensure feature cache exists (instantaneous if already generated)
%cd /content/deepsonar_v2
# If not already cached, generate the logmel cache for fold 0:
!python scripts/04_segment_and_cache.py --dataset deepship --features logmel

## 4. Run Linear Probing (Takes seconds on cached features)

In [ ]:
# Linear probe on Physics-Augmented MoCo
!python scripts/06_evaluate.py \
    --protocol linear_probe \
    --config configs/eval/linear_probe.yaml \
    --cache data/cache/deepship_fold_0_logmel \
    --checkpoint checkpoints/moco_physics/last.pt

In [ ]:
# Linear probe on Generic MoCo Baseline
!python scripts/06_evaluate.py \
    --protocol linear_probe \
    --config configs/eval/linear_probe.yaml \
    --cache data/cache/deepship_fold_0_logmel \
    --checkpoint checkpoints/moco_generic/last.pt

In [ ]:
# Linear probe on ImageNet-pretrained transfer baseline
!python scripts/06_evaluate.py \
    --protocol linear_probe \
    --config configs/eval/linear_probe.yaml \
    --cache data/cache/deepship_fold_0_logmel \
    --imagenet

## 5. Label Efficiency Curves (1% to 100% of labels, 3 seeds)

In [ ]:
# Physics-Augmented MoCo Label Efficiency
!python scripts/06_evaluate.py \
    --protocol label_efficiency \
    --config configs/eval/label_efficiency.yaml \
    --cache data/cache/deepship_fold_0_logmel \
    --checkpoint checkpoints/moco_physics/last.pt

In [ ]:
# Generic MoCo Baseline Label Efficiency
!python scripts/06_evaluate.py \
    --protocol label_efficiency \
    --config configs/eval/label_efficiency.yaml \
    --cache data/cache/deepship_fold_0_logmel \
    --checkpoint checkpoints/moco_generic/last.pt

## 6. Recording-ID Shuffle Leakage Control

In [ ]:
# Rule 2 Compliance: Recording-ID shuffle control
!python scripts/06_evaluate.py \
    --protocol leakage_control \
    --config configs/eval/linear_probe.yaml \
    --cache data/cache/deepship_fold_0_logmel \
    --checkpoint checkpoints/moco_physics/last.pt

## 7. Render Figures (Dual t-SNE, Grad-CAM, Label-Efficiency Curves)

In [ ]:
# Render headline figures
!python scripts/07_make_figures.py --label-efficiency
!python scripts/07_make_figures.py --embeddings --checkpoint checkpoints/moco_physics/last.pt
!python scripts/07_make_figures.py --gradcam --checkpoint checkpoints/moco_physics/last.pt

# Copy all generated tables and figures to Google Drive
!cp -r results/tables/* "/content/drive/MyDrive/deepsonar_v2_results/tables/" || true
!cp -r results/figures/* "/content/drive/MyDrive/deepsonar_v2_results/figures/" || true
print("All evaluation tables and figures saved to Google Drive!")